# 06 — Low-confidence mining and class-adaptive threshold calibration

**Project:** RAOD++ · **Phase:** F6 · **Protocol:** v1.1

This notebook does **not** generate crops yet. It answers the three questions that must
be settled before any crop is written to disk:

1. What does the confidence distribution of the E0 baseline look like, per class?
2. Where should the mining threshold τ sit, and is a class-adaptive τ justified by data?
3. Do low-confidence regions actually contain real objects, or just background noise?

Question 3 is the load-bearing one. If low-confidence regions are mostly noise, the whole
premise of RAOD++ collapses and we need to know that now, not after building the pipeline.

**Runtime:** ~25–40 min (inference over the training split for 10 seeds). GPU required.


## [Cell 3] Config

Two thresholds are involved and they must not be confused:

- **τ (mining threshold)** — the *upper* bound of the low-confidence band. Detections
  below it are treated as uncertain regions worth enriching. This is what we calibrate here.
- **σ (pseudo-label acceptance)** — a *high* threshold used later when retrieved crops
  enter training. Not touched in this notebook.

`FLOOR` excludes the noise tail: below it, YOLO outputs are dominated by background.

In [ ]:
# ===== Cell 3 =====
from pathlib import Path
import numpy as np, pandas as pd, yaml

CONFIG = {
    "PROJECT_ROOT": Path.home() / "RAODpp",
    "EXP": "E0",
    "DATASET": "visdrone",
    "SEEDS": [42, 123, 2024, 7, 3407, 11, 55, 101, 777, 12345],
    "IMGSZ": 640,
    "DEVICE": 0,
    "PREDICT_CONF": 0.01,      # inference floor; band analysis happens afterwards
    "FLOOR": 0.05,             # below this, detections are treated as background noise
    "PERCENTILES": [15, 25, 35],   # protocol v1.0 sweep; 25 is the default
    "DEFAULT_PCT": 25,
    "IOU_MATCH": 0.5,          # for the "is there a real object there?" analysis
    "ALPHA": 0.20,             # context margin for crops (used in notebook 07)
}

ROOT = CONFIG["PROJECT_ROOT"]
OUT = ROOT/"output"; FIGDATA = OUT/"figdata"; FIGDATA.mkdir(parents=True, exist_ok=True)
RUNS = ROOT/"model"/"runs"
DATA_YAML = ROOT/"dataset"/"visdrone_raodpp.yaml"

dcfg = yaml.safe_load(open(DATA_YAML))
BASE = Path(dcfg["path"]); NAMES = dcfg["names"]
TRAIN_IMG = BASE/dcfg["train"]
TRAIN_LBL = Path(str(TRAIN_IMG).replace("/images","/labels"))

print("train images:", len(list(TRAIN_IMG.glob('*.jpg'))))
print("classes     :", len(NAMES))
print("seeds       :", CONFIG["SEEDS"])


## [Cell 6] A note on where the mining model has been

Predictions here are made by each E0 model **on its own training split**. Confidences are
therefore inflated relative to unseen data — the model has memorised much of this content.

We use this deliberately rather than working around it. A region the detector is *still*
unsure about after training on it is, by construction, a harder case than one it merely
finds unfamiliar. The manuscript states this explicitly instead of leaving it for a
reviewer to discover. A seed-crossed variant (mine with seed A, train seed B) is kept as a
robustness check for the ablation phase.

## [Cell 6] Inference over the training split
One pass per seed, results cached to `figdata/train_predictions_<run>.csv`. Re-running
skips seeds that are already done.

In [ ]:
# ===== Cell 6 =====
from ultralytics import YOLO
import time

def predict_train(seed):
    run_id = f"{CONFIG['EXP']}_{CONFIG['DATASET']}_s{seed}"
    out_csv = FIGDATA/f"train_predictions_{run_id}.csv"
    if out_csv.exists():
        print(f"  {run_id}: cached ({len(pd.read_csv(out_csv)):,} rows)")
        return out_csv
    w = RUNS/run_id/"weights"/"best.pt"
    assert w.exists(), f"missing {w}"
    model = YOLO(str(w))
    imgs = [str(p) for p in sorted(TRAIN_IMG.glob("*.jpg"))]
    rows = []
    t0 = time.time()
    for r in model.predict(source=imgs, imgsz=CONFIG["IMGSZ"], conf=CONFIG["PREDICT_CONF"],
                           device=CONFIG["DEVICE"], stream=True, verbose=False):
        b = r.boxes
        if b is None or len(b) == 0:
            continue
        name = Path(r.path).name
        xyxy = b.xyxy.cpu().numpy(); cls = b.cls.cpu().numpy(); cf = b.conf.cpu().numpy()
        for k in range(len(cf)):
            rows.append((name, int(cls[k]), float(cf[k]),
                         float(xyxy[k][0]), float(xyxy[k][1]),
                         float(xyxy[k][2]), float(xyxy[k][3])))
    df = pd.DataFrame(rows, columns=["image","class_id","conf","x1","y1","x2","y2"])
    df.to_csv(out_csv, index=False)
    print(f"  {run_id}: {len(df):,} detections in {(time.time()-t0)/60:.1f} min")
    return out_csv

paths = [predict_train(s) for s in CONFIG["SEEDS"]]
print("\ndone:", len(paths), "seeds")


## [Cell 8] Confidence distribution per class
Establishes whether one global τ can serve every class. If the per-class distributions sit
at visibly different levels, a single threshold systematically over-mines some classes and
under-mines others — which is exactly the argument the adaptive-threshold literature makes
(CascadeMatch's APM, FreeMatch's class-local thresholds).

In [ ]:
# ===== Cell 8 =====
ref_seed = CONFIG["SEEDS"][0]
ref_id = f"{CONFIG['EXP']}_{CONFIG['DATASET']}_s{ref_seed}"
det = pd.read_csv(FIGDATA/f"train_predictions_{ref_id}.csv")
det = det[det.conf >= CONFIG["FLOOR"]]
det["class_name"] = det.class_id.map(NAMES)

stats = (det.groupby(["class_id","class_name"]).conf
         .agg(n="size", mean="mean", median="median",
              p15=lambda s: s.quantile(.15), p25=lambda s: s.quantile(.25),
              p35=lambda s: s.quantile(.35), p75=lambda s: s.quantile(.75))
         .round(4).reset_index())
stats.to_csv(FIGDATA/"conf_distribution_per_class.csv", index=False)
print(stats.to_string(index=False))
print(f"\nglobal p25 = {det.conf.quantile(.25):.4f}   (single fixed threshold would use this)")
print(f"spread of per-class p25: {stats.p25.min():.3f} - {stats.p25.max():.3f}")


## [Cell 10] Class-adaptive τ, averaged over seeds
τ for class c is the p-th percentile of that class's confidence distribution, averaged
across the 10 baseline models so the threshold does not depend on one seed's quirks.
Computed for all three percentiles in the protocol sweep.

In [ ]:
# ===== Cell 10 =====
rows = []
for seed in CONFIG["SEEDS"]:
    rid = f"{CONFIG['EXP']}_{CONFIG['DATASET']}_s{seed}"
    d = pd.read_csv(FIGDATA/f"train_predictions_{rid}.csv")
    d = d[d.conf >= CONFIG["FLOOR"]]
    for c, g in d.groupby("class_id"):
        for p in CONFIG["PERCENTILES"]:
            rows.append({"seed": seed, "class_id": int(c), "pct": p,
                         "tau": float(g.conf.quantile(p/100))})

tau_long = pd.DataFrame(rows)
tau = (tau_long.groupby(["class_id","pct"]).tau.agg(["mean","std"]).reset_index()
       .rename(columns={"mean":"tau","std":"tau_std"}))
tau["class_name"] = tau.class_id.map(NAMES)
tau.to_csv(FIGDATA/"tau_class_adaptive.csv", index=False)

wide = tau.pivot(index="class_name", columns="pct", values="tau").round(4)
print("class-adaptive tau by percentile (mean over 10 seeds)")
print(wide.to_string())
print(f"\nseed-to-seed std of tau: max = {tau.tau_std.max():.4f}  (stability check)")


## [Cell 12] Does the low-confidence band contain real objects?

The premise test. For the reference seed we take every detection inside the band
[FLOOR, τ_class] and ask whether it overlaps a ground-truth box at IoU ≥ 0.5, and whether
that ground-truth object was *already* covered by a confident detection. The interesting
quantity is the third row: real objects that only the low-confidence band caught.

In [ ]:
# ===== Cell 12 =====
def iou_matrix(a, b):
    """a: (N,4), b: (M,4) in xyxy -> (N,M) IoU"""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:,None,0], b[None,:,0]); y1 = np.maximum(a[:,None,1], b[None,:,1])
    x2 = np.minimum(a[:,None,2], b[None,:,2]); y2 = np.minimum(a[:,None,3], b[None,:,3])
    inter = np.clip(x2-x1, 0, None) * np.clip(y2-y1, 0, None)
    ar_a = (a[:,2]-a[:,0]) * (a[:,3]-a[:,1]); ar_b = (b[:,2]-b[:,0]) * (b[:,3]-b[:,1])
    return inter / (ar_a[:,None] + ar_b[None,:] - inter + 1e-9)

from PIL import Image
tau_map = dict(zip(tau[tau.pct==CONFIG["DEFAULT_PCT"]].class_id,
                   tau[tau.pct==CONFIG["DEFAULT_PCT"]].tau))

det_all = pd.read_csv(FIGDATA/f"train_predictions_{ref_id}.csv")
sample_images = sorted(det_all.image.unique())[:1500]      # subsample for speed
n_band = n_band_hit = n_band_only = n_conf = 0

for img in sample_images:
    d = det_all[det_all.image == img]
    lp = TRAIN_LBL/(Path(img).stem + ".txt")
    if not lp.exists():
        continue
    W, H = Image.open(TRAIN_IMG/img).size
    gt = []
    for line in open(lp):
        f = line.split()
        if len(f) != 5: continue
        c = int(f[0]); cx, cy, bw, bh = map(float, f[1:])
        gt.append([(cx-bw/2)*W, (cy-bh/2)*H, (cx+bw/2)*W, (cy+bh/2)*H, c])
    if not gt: continue
    gt = np.array(gt)

    d = d.assign(tau=d.class_id.map(tau_map).fillna(0.3))
    band = d[(d.conf >= CONFIG["FLOOR"]) & (d.conf < d.tau)]
    conf = d[d.conf >= d.tau]
    n_band += len(band); n_conf += len(conf)

    if len(band):
        m = iou_matrix(band[["x1","y1","x2","y2"]].values, gt[:, :4])
        hit = m.max(axis=1) >= CONFIG["IOU_MATCH"]
        n_band_hit += int(hit.sum())
        if len(conf):
            mc = iou_matrix(conf[["x1","y1","x2","y2"]].values, gt[:, :4])
            covered = (mc >= CONFIG["IOU_MATCH"]).any(axis=0)     # GT already detected
            best_gt = m.argmax(axis=1)
            n_band_only += int(((m.max(axis=1) >= CONFIG["IOU_MATCH"]) &
                                (~covered[best_gt])).sum())
        else:
            n_band_only += int(hit.sum())

res = pd.DataFrame([
    ["images analysed", len(sample_images)],
    ["confident detections (conf >= tau)", n_conf],
    ["low-confidence band detections", n_band],
    ["band detections matching a GT object (IoU>=0.5)", n_band_hit],
    ["band precision (share hitting real objects)", round(n_band_hit/max(1,n_band), 4)],
    ["GT objects recovered ONLY by the band", n_band_only],
], columns=["quantity","value"])
res.to_csv(FIGDATA/"lowconf_band_analysis.csv", index=False)
print(res.to_string(index=False))


## [Cell 14] Crop budget: what each design option would cost

Three ways to build the crop set, with different rigour/cost trade-offs. The numbers below
are measured, not guessed, so the choice can be made on evidence.

In [ ]:
# ===== Cell 14 =====
per_seed = []
for seed in CONFIG["SEEDS"]:
    rid = f"{CONFIG['EXP']}_{CONFIG['DATASET']}_s{seed}"
    d = pd.read_csv(FIGDATA/f"train_predictions_{rid}.csv")
    d = d.assign(tau=d.class_id.map(tau_map).fillna(0.3))
    n = int(((d.conf >= CONFIG["FLOOR"]) & (d.conf < d.tau)).sum())
    per_seed.append({"seed": seed, "band_crops": n})

ps = pd.DataFrame(per_seed)
mean_crops = ps.band_crops.mean()
KB = 15  # ~15 KB per 224x224 JPEG crop

opts = pd.DataFrame([
    ["A: seed-matched (each seed mines its own)", mean_crops*len(CONFIG["SEEDS"]),
     "most rigorous; 10x CLIP + storage"],
    ["B: single mining model (seed 42)", mean_crops,
     "cheapest; enrichment tied to one seed"],
    ["C: consensus across seeds", mean_crops,
     "one crop set, robust; needs cross-model box matching"],
], columns=["option","crops","note"])
opts["storage_GB"] = (opts.crops*KB/1e6).round(2)
opts["clip_minutes"] = (opts.crops/8000).round(1)   # ~8k crops/min on RTX 4500 (fp16)

print(ps.to_string(index=False))
print(f"\nmean band crops per seed: {mean_crops:,.0f}")
print()
print(opts.to_string(index=False))
opts.to_csv(FIGDATA/"crop_budget_options.csv", index=False)


**Decision point.** Report back: the per-class τ table, the band analysis (especially
*band precision* and *GT objects recovered only by the band*), and the crop budget. The
crop-set option (A / B / C) is chosen from these numbers, then notebook 07 writes the
crops and builds the CLIP + FAISS index.